In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

DATA = Path("__file__").resolve().parent.parent / "data" / "processed"
df = pd.read_csv(DATA / "fact_orders.csv", parse_dates=[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
])
df["purchase_month"] = df["order_purchase_timestamp"].dt.to_period("M")
print(f"{len(df):,} rows  |  {df.shape[1]} cols")
df.head()

## 1 — Monthly Revenue Trend

In [ ]:
monthly = df.groupby("purchase_month")["order_revenue"].sum()
ax = monthly.plot(kind="bar", color="steelblue", edgecolor="white")
ax.set_title("Monthly Revenue Trend", fontsize=14, weight="bold")
ax.set_ylabel("Revenue (R$)")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 2 — SLA Breach Rate by Month

In [ ]:
breach = df.groupby("purchase_month")["sla_breach"].mean() * 100
ax = breach.plot(kind="line", marker="o", color="crimson", linewidth=2)
ax.set_title("SLA Breach Rate by Month", fontsize=14, weight="bold")
ax.set_ylabel("Breach Rate (%)")
ax.set_xlabel("")
ax.axhline(breach.mean(), ls="--", color="gray", label=f"avg {breach.mean():.1f}%")
ax.legend()
plt.tight_layout()
plt.show()

## 3 — Delivery Days Distribution

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df["delivery_days"].dropna(), bins=50, kde=True, color="teal", ax=ax)
ax.set_title("Delivery Days Distribution", fontsize=14, weight="bold")
ax.set_xlabel("Days")
med = df["delivery_days"].median()
ax.axvline(med, color="red", ls="--", label=f"median = {med:.0f}d")
ax.legend()
plt.tight_layout()
plt.show()

## 4 — Top 10 Customers by Revenue

In [ ]:
top10 = (
    df.groupby("customer_id")["order_revenue"]
    .sum()
    .nlargest(10)
    .sort_values()
)
ax = top10.plot(kind="barh", color="darkorange", edgecolor="white")
ax.set_title("Top 10 Customers by Revenue", fontsize=14, weight="bold")
ax.set_xlabel("Revenue (R$)")
ax.set_ylabel("")
ax.set_yticklabels([cid[:8] + "\u2026" for cid in top10.index])
plt.tight_layout()
plt.show()

## 5 — Correlation Heatmap

In [ ]:
num_cols = ["order_revenue", "item_count", "delivery_days", "sla_breach"]
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, square=True, ax=ax)
ax.set_title("Correlation Heatmap", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()